# Milestone 2: training geometry diversity

Train fresh weights on the same direct shape-color task, expanding training from **16 to 128 geometries**. Retain the original 16 layouts, add 112 with broader coordinate and patch-offset coverage, and preserve the exact validation/test records and original size proportions.

Enable a GPU and internet, attach the completed `milestone2_shape_grounding_artifacts.zip`, and push these changes before running. Set `REFERENCE_SOURCE` below. This uses one GPU (`cuda:0`) and preserves Kaggle's PyTorch installation. Training runs for exactly **2,880 updates / 92,160 QA presentations**.

The architecture, renderer, questions and optimizer are unchanged. Each QA appears once or twice, averaging 1⅔ presentations versus 13⅓ previously. Generation and training expose coverage and presentation reports; final diagnostics separate original and added layouts and compare the same validation set against the reference. No reference weights initialize the model, and no test examples are evaluated.

See `docs/milestones/milestone2_geometry_diversity.md` for the protocol and interpretation limits. A fresh output directory is required; training resume is not supported.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # A committed revision containing this notebook; commit SHA preferred.
REPO_DIR = "/kaggle/working/multi-modal-loop-geometry-diversity"
RUN_ROOT = "/kaggle/working/milestone2_geometry_diversity"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_shape_grounding_artifacts.zip"
)

## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit the source and record the protocol

The completed shape-grounding archive is audited before generation. Its manifest fixes the original layouts and held-out splits. Fresh weights are used; reference weights are never used for initialization.


In [ ]:
import multimodal_loop.eval.kaggle_geometry_diversity as helpers
from multimodal_loop.eval.kaggle_geometry_diversity import (
    archive_geometry_diversity,
    prepare_geometry_diversity,
    run_geometry_diversity,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_geometry_diversity(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE)

## Train and evaluate

Generate and validate coordinate/patch-offset coverage, then train for one full pass plus 1,152 batches of pass two. No early stopping or checkpoint selection. Frozen evaluation covers 55,296 training and the same 1,728 validation questions, with original/added training layouts reported separately. Controls keep recipient targets fixed. Logs stream below and are retained.


In [ ]:
report = run_geometry_diversity(run)

## Inspect the results

Training criteria: at least 95% accuracy for each shape. Held-out direct grounding: at least 90% for each shape and 30 percentage-point overall gaps against blank images and both mean shuffle controls. These are diagnostic criteria, not Milestone 2 completion gates.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

for split, metrics in report["splits"].items():
    print(split, {k: metrics[k] for k in ("total", "accuracy", "loss", "all_three")})
    print("Shape accuracy:", metrics["breakdowns"]["shape"])
print(
    "Training subsets:",
    {
        name: {k: m[k] for k in ("total", "accuracy", "loss")}
        for name, m in report["training_geometry_subsets"].items()
    },
)
print(json.dumps(report["assessment"], indent=2))
comparison = json.loads((run.root / "diagnosis" / "comparison.json").read_text())
print(json.dumps(comparison["validation"], indent=2))
coverage = json.loads((run.root / "data" / "coverage.json").read_text())
print(
    "Coverage:",
    {
        name: {k: coverage[name][k] for k in ("original", "selected", "attainable")}
        for name in ("coordinates", "patch_offsets")
    },
)
presentations = json.loads((run.root / "data" / "presentations.json").read_text())
print(
    "Presentations:", {name: value["qa_visit_histogram"] for name, value in presentations.items()}
)
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))

## Retain artifacts

Download the archive and bring it back for review. It includes the checkpoint, manifest, protocol, metrics, all train/validation predictions, rendered error preview, controls, hashes, runtime/revision provenance and logs. Staged reference weights are excluded; manifests, derivation, coverage, presentation counts and reference comparison are retained.


In [ ]:
archive = archive_geometry_diversity(run)
print("Geometry-diversity archive:", archive)
display(FileLink(str(archive)))